In [1]:
!pip install autogluon.tabular

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 514.8/514.8 kB 15.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 248.4/248.4 kB 27.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 112.3/112.3 kB 13.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 90.7/90.7 kB 10.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 140.0/140.0 kB 15.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 15.7/15.7 MB 103.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 48.9/48.9 MB 17.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 90.2/90.2 kB 10.5 MB/s eta 0:00:00
  Attempting uninstall: pyarrow
    Found existing installation: pyarrow 18.1.0
    Uninstalling pyarrow-18.1.0:
      Successfully uninstalled pyarrow-18.1.0


In [2]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [3]:
import os
import pandas as pd
import numpy as np
from pathlib import Path

import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms

In [4]:
datasetPath = Path("/content/drive/MyDrive/ML-For-CV-Robustness/datasets/datasetComp.csv")

#making sure
datasetPath.is_file()

True

In [6]:
dataset = pd.read_csv(datasetPath)
dataset

,grviIn,meanRefR,meanRefG,illumR,illumG,blockIdx,sunRoll,sunPitch,sunYaw,camRoll,camPitch,camYaw,grviOut
0,0.163549,0.000324,0.000421,1130.973355,1020.232214,0.0,180.0,-63.812729,-56.074120,0.0,-19.258,-0.0,0.088270
1,0.097110,0.000412,0.000486,1130.973355,1020.232214,1.0,180.0,-63.812729,-56.074120,0.0,-19.258,-0.0,0.057202
2,0.074345,0.000448,0.000517,1130.973355,1020.232214,2.0,180.0,-63.812729,-56.074120,0.0,-19.258,-0.0,0.048503
3,0.156574,0.000374,0.000481,1130.973355,1020.232214,3.0,180.0,-63.812729,-56.074120,0.0,-19.258,-0.0,0.086477
4,0.116627,0.000393,0.000479,1130.973355,1020.232214,4.0,180.0,-63.812729,-56.074120,0.0,-19.258,-0.0,0.067095
...,...,...,...,...,...,...,...,...,...,...,...,...,...
667235,0.261726,0.000240,0.000362,553.313000,483.020000,15.0,180.0,-42.301559,-85.835487,-0.0,-22.218,-180.0,0.087821
667236,0.239247,0.000270,0.000396,553.313000,483.020000,16.0,180.0,-42.301559,-85.835487,-0.0,-22.218,-180.0,0.079927
667237,0.283672,0.000269,0.000422,553.313000,483.020000,17.0,180.0,-42.301559,-85.835487,-0.0,-22.218,-180.0,0.097174
667238,0.241322,0.000345,0.000508,553.313000,483.020000,18.0,180.0,-42.301559,-85.835487,-0.0,-22.218,-180.0,0.090264


In [7]:
from autogluon.tabular import TabularPredictor

In [8]:
train_df = dataset.sample(frac=0.8, random_state=42)
test_df = dataset.drop(train_df.index)

In [9]:
train_df

,grviIn,meanRefR,meanRefG,illumR,illumG,blockIdx,sunRoll,sunPitch,sunYaw,camRoll,camPitch,camYaw,grviOut
239397,0.255627,0.000572,0.000916,262.126637,200.472881,17.0,180.0,-19.763107,-104.398338,-0.0,-19.620,-0.00,0.125760
460616,0.291634,0.000310,0.000475,334.972000,302.575000,16.0,180.0,-63.812729,-56.074120,0.0,-22.440,-147.65,0.095608
193058,0.280101,0.000411,0.000702,1336.747674,1022.588409,18.0,180.0,-19.798325,104.358398,-0.0,-19.278,-0.00,0.142003
555725,0.811764,0.000306,0.000472,223.838000,202.044000,5.0,180.0,-63.688171,56.363342,-0.0,-22.038,-180.00,0.082971
335353,0.180071,0.029525,0.037890,1.540000,2.103000,13.0,180.0,-63.812729,-56.074120,0.0,-22.053,180.00,0.093083
...,...,...,...,...,...,...,...,...,...,...,...,...,...
539354,0.188815,0.000378,0.000529,1174.170000,1024.945000,14.0,180.0,-42.301559,-85.835487,-0.0,-19.350,-0.00,0.092095
651738,0.152599,0.041819,0.054116,2.088000,2.497000,18.0,180.0,-19.798325,104.358398,-0.0,-19.250,-0.00,0.147274
512486,0.205018,0.000227,0.000300,1138.042000,1026.515000,6.0,180.0,-63.688171,56.363342,-0.0,-19.670,-0.00,0.078602
185954,0.197553,0.000359,0.000502,1165.530874,1015.519825,14.0,180.0,-42.151062,85.965363,-0.0,-19.476,-0.00,0.098553


In [10]:
test_df

,grviIn,meanRefR,meanRefG,illumR,illumG,blockIdx,sunRoll,sunPitch,sunYaw,camRoll,camPitch,camYaw,grviOut
3,0.156574,0.000374,0.000481,1130.973355,1020.232214,3.0,180.0,-63.812729,-56.074120,0.0,-19.258,-0.0,0.086477
5,0.081775,0.000463,0.000538,1130.973355,1020.232214,5.0,180.0,-63.812729,-56.074120,0.0,-19.258,-0.0,0.053121
8,0.132084,0.000445,0.000558,1130.973355,1020.232214,8.0,180.0,-63.812729,-56.074120,0.0,-19.258,-0.0,0.079018
13,0.108716,0.000371,0.000444,1130.973355,1020.232214,13.0,180.0,-63.812729,-56.074120,0.0,-19.258,-0.0,0.066583
15,0.144419,0.000328,0.000411,1130.973355,1020.232214,15.0,180.0,-63.812729,-56.074120,0.0,-19.258,-0.0,0.087779
...,...,...,...,...,...,...,...,...,...,...,...,...,...
667222,0.258746,0.000165,0.000226,553.313000,483.020000,2.0,180.0,-42.301559,-85.835487,-0.0,-22.218,-180.0,0.093556
667223,0.261627,0.000149,0.000207,553.313000,483.020000,3.0,180.0,-42.301559,-85.835487,-0.0,-22.218,-180.0,0.095516
667224,0.289089,0.000165,0.000243,553.313000,483.020000,4.0,180.0,-42.301559,-85.835487,-0.0,-22.218,-180.0,0.104448
667234,0.251055,0.000267,0.000392,553.313000,483.020000,14.0,180.0,-42.301559,-85.835487,-0.0,-22.218,-180.0,0.085302


In [11]:
predictor = TabularPredictor(
    label="grviOut",
    problem_type="regression",
    eval_metric="root_mean_squared_error"
).fit(
    train_data=train_df
)

No path specified. Models will be saved in: "AutogluonModels/ag-20260823_131346"
Verbosity: 2 (Standard Logging)
=================== System Info ===================
AutoGluon Version:  1.6.1
Python Version:     3.13.15
Operating System:   Linux
Platform Machine:   x86_64
Platform Version:   #1 SMP Thu Apr 30 18:17:14 UTC 2026
CPU Count:          2
Pytorch Version:    2.11.0+cu128
CUDA Version:       12.8
GPU Memory:         GPU 0: 14.56/14.56 GB
Total GPU Memory:   Free: 14.56 GB, Allocated: 0.00 GB, Total: 14.56 GB
GPU Count:          1
Memory Avail:       10.61 GB / 12.67 GB (83.7%)
Disk Space Avail:   65.03 GB / 112.64 GB (57.7%)
No presets specified! To achieve strong results with AutoGluon, it is recommended to use the available presets. Defaulting to `'medium'`...
	Recommended Presets (For more details refer to https://auto.gluon.ai/stable/tutorials/tabular/tabular-essentials.html#presets):
	presets='extreme'  : Use this if you have a GPU. The go-to preset for best results, and t

[1000]	valid_set's rmse: 0.00835794
[2000]	valid_set's rmse: 0.00775006
[3000]	valid_set's rmse: 0.00742337
[4000]	valid_set's rmse: 0.00721191
[5000]	valid_set's rmse: 0.00705085
[6000]	valid_set's rmse: 0.00693653
[7000]	valid_set's rmse: 0.00683949
[8000]	valid_set's rmse: 0.00675056
[9000]	valid_set's rmse: 0.00668076
[10000]	valid_set's rmse: 0.00661738


	-0.0066	 = Validation score   (-root_mean_squared_error)
	277.72s	 = Training   runtime
	5.19s	 = Validation runtime
Fitting model: LightGBM ...
	Fitting with cpus=1, gpus=0, mem=0.3/10.4 GB


[1000]	valid_set's rmse: 0.00753361
[2000]	valid_set's rmse: 0.00687932
[3000]	valid_set's rmse: 0.00653599
[4000]	valid_set's rmse: 0.00630254
[5000]	valid_set's rmse: 0.00613705
[6000]	valid_set's rmse: 0.00601045
[7000]	valid_set's rmse: 0.00588689
[8000]	valid_set's rmse: 0.00579453
[9000]	valid_set's rmse: 0.00572182
[10000]	valid_set's rmse: 0.00563807


	-0.0056	 = Validation score   (-root_mean_squared_error)
	197.59s	 = Training   runtime
	3.88s	 = Validation runtime
Fitting model: RandomForestMSE ...
	Fitting with cpus=2, gpus=0, mem=2.6/10.4 GB
	-0.0076	 = Validation score   (-root_mean_squared_error)
	1092.13s	 = Training   runtime
	0.78s	 = Validation runtime
Fitting model: CatBoost ...
	Fitting with cpus=1, gpus=0
		`import catboost` failed. A quick tip is to install via `pip install autogluon.tabular[catboost]==1.6.1`.
Fitting model: ExtraTreesMSE ...
	Fitting with cpus=2, gpus=0, mem=2.6/10.3 GB
	-0.0065	 = Validation score   (-root_mean_squared_error)
	183.48s	 = Training   runtime
	0.38s	 = Validation runtime
Fitting model: NeuralNetFastAI ...
	Fitting with cpus=1, gpus=0, mem=0.4/10.3 GB
No improvement since epoch 7: early stopping
	-0.0129	 = Validation score   (-root_mean_squared_error)
	305.81s	 = Training   runtime
	0.06s	 = Validation runtime
Fitting model: XGBoost ...
	Fitting with cpus=1, gpus=0
	-0.0055	 = Validati

[1000]	valid_set's rmse: 0.00673628
[2000]	valid_set's rmse: 0.00618086
[3000]	valid_set's rmse: 0.00589164
[4000]	valid_set's rmse: 0.00568662
[5000]	valid_set's rmse: 0.0055419
[6000]	valid_set's rmse: 0.00541843
[7000]	valid_set's rmse: 0.00530799
[8000]	valid_set's rmse: 0.00521889
[9000]	valid_set's rmse: 0.00513438
[10000]	valid_set's rmse: 0.0050749


	-0.0051	 = Validation score   (-root_mean_squared_error)
	219.83s	 = Training   runtime
	4.41s	 = Validation runtime
Fitting model: WeightedEnsemble_L2 ...
	Fitting 1 model on all data | Fitting with cpus=2, gpus=0, mem=0.0/9.8 GB
	Ensemble Weights: {'LightGBMLarge': 0.625, 'ExtraTreesMSE': 0.25, 'XGBoost': 0.125}
	-0.0048	 = Validation score   (-root_mean_squared_error)
	0.03s	 = Training   runtime
	0.0s	 = Validation runtime
AutoGluon training complete, total runtime = 3517.26s ... Best model: WeightedEnsemble_L2 | Estimated inference throughput: 979.4 rows/s (5338 batch size)
TabularPredictor saved. To load, use: predictor = TabularPredictor.load("/content/AutogluonModels/ag-20260823_131346")


In [12]:
predictor.leaderboard()

,model,score_val,eval_metric,pred_time_val,fit_time,pred_time_val_marginal,fit_time_marginal,stack_level,can_infer,fit_order
0,WeightedEnsemble_L2,-0.004849,root_mean_squared_error,5.450155,658.312209,0.000722,0.029097,2,True,9
1,LightGBMLarge,-0.005075,root_mean_squared_error,4.405904,219.827334,4.405904,219.827334,1,True,8
2,XGBoost,-0.005457,root_mean_squared_error,0.661669,254.971510,0.661669,254.971510,1,True,6
3,LightGBM,-0.005638,root_mean_squared_error,3.884629,197.594816,3.884629,197.594816,1,True,2
4,ExtraTreesMSE,-0.006450,root_mean_squared_error,0.381860,183.484268,0.381860,183.484268,1,True,4
5,LightGBMXT,-0.006617,root_mean_squared_error,5.187049,277.720214,5.187049,277.720214,1,True,1
6,RandomForestMSE,-0.007578,root_mean_squared_error,0.781628,1092.134446,0.781628,1092.134446,1,True,3
7,NeuralNetTorch,-0.007617,root_mean_squared_error,0.034087,949.473123,0.034087,949.473123,1,True,7
8,NeuralNetFastAI,-0.012947,root_mean_squared_error,0.059475,305.807295,0.059475,305.807295,1,True,5


In [13]:
predictor.leaderboard(test_df, silent=True)

,model,score_test,score_val,eval_metric,pred_time_test,pred_time_val,fit_time,pred_time_test_marginal,pred_time_val_marginal,fit_time_marginal,stack_level,can_infer,fit_order
0,WeightedEnsemble_L2,-0.004989,-0.004849,root_mean_squared_error,138.495898,5.450155,658.312209,0.018813,0.000722,0.029097,2,True,9
1,LightGBMLarge,-0.005211,-0.005075,root_mean_squared_error,114.555724,4.405904,219.827334,114.555724,4.405904,219.827334,1,True,8
2,XGBoost,-0.005672,-0.005457,root_mean_squared_error,18.638642,0.661669,254.971510,18.638642,0.661669,254.971510,1,True,6
3,LightGBM,-0.005806,-0.005638,root_mean_squared_error,86.722501,3.884629,197.594816,86.722501,3.884629,197.594816,1,True,2
4,ExtraTreesMSE,-0.006538,-0.006450,root_mean_squared_error,5.282719,0.381860,183.484268,5.282719,0.381860,183.484268,1,True,4
5,LightGBMXT,-0.006873,-0.006617,root_mean_squared_error,127.157247,5.187049,277.720214,127.157247,5.187049,277.720214,1,True,1
6,NeuralNetTorch,-0.007811,-0.007617,root_mean_squared_error,0.864791,0.034087,949.473123,0.864791,0.034087,949.473123,1,True,7
7,RandomForestMSE,-0.007934,-0.007578,root_mean_squared_error,6.793788,0.781628,1092.134446,6.793788,0.781628,1092.134446,1,True,3
8,NeuralNetFastAI,-0.013072,-0.012947,root_mean_squared_error,0.885033,0.059475,305.807295,0.885033,0.059475,305.807295,1,True,5
